### **Data Cleaning**

In [1]:
df_bronze = spark.read.format("csv").option("header", "true").load("Files/COVID_19.csv")
display(df_bronze)

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 16b87896-d070-4e56-9794-e8a1b0a3d5dc)

### Handle Duplicates

In [2]:
from pyspark.sql.functions import col

# check for duplicates (date + country combine)
duplicates = (
    df_bronze
        .groupBy("dateRep", "countriesAndTerritories", "continentExp")
        .count()
        .filter(col("count") > 1)
)

if duplicates.count() > 0:
    print("Duplicate records found for the following date-country combinations:")
    duplicates.show()
else:
    print("No duplicate records found")

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 6, Finished, Available, Finished, False)

No duplicate records found


In [3]:
df_bronze.count()

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 7, Finished, Available, Finished, False)

61900

In [4]:
# store raw data as delta table in the bronze layer
df_bronze.write.format("delta").mode("overwrite").save("Tables/dbo/covid_bronze_raw_data")

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 8, Finished, Available, Finished, False)

### Handle Null values

In [5]:
# Assuming df_bronze is the Dataframe for the bronze layer data

# Get a list of columns in the Dataframe
columns = df_bronze.columns

# Loop through each column and check for null values
for column in columns:
    null_count = df_bronze.filter(df_bronze[column].isNull()).count()
    print(f"Column '{column}' has {null_count} null values.")

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 9, Finished, Available, Finished, False)

Column 'dateRep' has 0 null values.
Column 'day' has 0 null values.
Column 'month' has 0 null values.
Column 'year' has 0 null values.
Column 'cases' has 0 null values.
Column 'deaths' has 0 null values.
Column 'countriesAndTerritories' has 0 null values.
Column 'geoId' has 275 null values.
Column 'countryterritoryCode' has 123 null values.
Column 'popData2019' has 123 null values.
Column 'continentExp' has 0 null values.
Column 'Cumulative_number_for_COVID_19_cases' has 2879 null values.


In [6]:
# Drop the 'popData2019' and 'Cumulative_number_for_COVID_19_cases' columns
df_bronze_cleaned = df_bronze.drop('popData2019', 'Cumulative_number_for_COVID_19_cases')

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 10, Finished, Available, Finished, False)

In [7]:
# Replace nulls in the 'countryterritoryCode' column with 'unknown'
df_bronze_cleaned = df_bronze_cleaned.fillna({'countryterritoryCode': 'Unknown'})

# Replace nulls in the 'geoId' column with 'unknown'
df_bronze_cleaned = df_bronze_cleaned.fillna({'geoId': 'Unknown'})

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 11, Finished, Available, Finished, False)

In [8]:
# Assuming df_bronze is the Dataframe for the bronze layer data

# Get a list of columns in the Dataframe
columns = df_bronze_cleaned.columns

# Loop through each column and check for null values
for column in columns:
    null_count = df_bronze_cleaned.filter(df_bronze_cleaned[column].isNull()).count()
    print(f"Column '{column}' has {null_count} null values.")

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 12, Finished, Available, Finished, False)

Column 'dateRep' has 0 null values.
Column 'day' has 0 null values.
Column 'month' has 0 null values.
Column 'year' has 0 null values.
Column 'cases' has 0 null values.
Column 'deaths' has 0 null values.
Column 'countriesAndTerritories' has 0 null values.
Column 'geoId' has 0 null values.
Column 'countryterritoryCode' has 0 null values.
Column 'continentExp' has 0 null values.


### **Tranformation**

### Transform data and store in silver layer

In [9]:
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("countriesAndTerritories").orderBy("year", "month", "day")

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 13, Finished, Available, Finished, False)

In [10]:
from pyspark.sql.functions import sum, to_date, col

# Calculate cumulative cases
df_silver = df_bronze_cleaned.withColumn("cumulative_cases", sum("cases").over(windowSpec))

# Calculate cumulative deaths
df_silver = df_silver.withColumn("cumulative_deaths", sum("deaths").over(windowSpec))

# Covert dateRep type to date 
df_silver = df_silver.withColumn("dateRep", to_date(col("dateRep"), "dd-MM-yyyy"))

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 14, Finished, Available, Finished, False)

In [11]:
display(df_silver)

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 60cfb8d7-4b7a-446e-81c2-cac5f7f8d534)

In [12]:
# Register the DataFrame as a temporary SQL table
df_silver.createOrReplaceTempView("covid_data")

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 16, Finished, Available, Finished, False)

In [13]:
%%sql
SELECT * FROM covid_data WHERE countriesAndTerritories = 'Australia';

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 17, Finished, Available, Finished, False)

<Spark SQL result set with 350 rows and 12 fields>

In [14]:
%%sql
SELECT 
    DISTINCT continentExp,
    countriesAndTerritories
FROM covid_data;

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 18, Finished, Available, Finished, False)

<Spark SQL result set with 214 rows and 2 fields>

### Aggregate data by time periods 

In [15]:
from pyspark.sql import functions as F

# Aggregate data by day
daily_agg = (
    df_silver.groupBy("daterep", "year", "month", "day", "countriesAndTerritories", "continentExp")
        .agg(
            F.sum("cases").alias("total_daily_cases"),
            F.sum("deaths").alias("total_daily_deaths")
        )
)

# Aggregate data by week (using date_trunc to truncate the date to the week start)
weekly_agg = (
    df_silver.withColumn("week", F.date_trunc("week",  F.col("daterep")))
        .groupBy("week", "countriesAndTerritories")
        .agg(
            F.sum("cases").alias("total_weekly_cases"),
            F.sum("deaths").alias("total_weekly_deaths")
        )
)

# Aggregate data by month (using date_trunc to truncate the date to the month start)
monthy_agg = (
    df_silver.withColumn("month", F.date_trunc("month", F.col("daterep")))
        .groupBy("month", "countriesAndTerritories")
        .agg(
            F.sum("cases").alias("total_montly_cases"),
            F.sum("deaths").alias("total_montly_deaths")
        )
)

# show aggregated data
daily_agg.show()
weekly_agg.show()
monthy_agg.show()

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 19, Finished, Available, Finished, False)

+----------+----+-----+---+-----------------------+------------+-----------------+------------------+
|   daterep|year|month|day|countriesAndTerritories|continentExp|total_daily_cases|total_daily_deaths|
+----------+----+-----+---+-----------------------+------------+-----------------+------------------+
|2020-06-21|2020|    6| 21|            Afghanistan|        Asia|            546.0|              21.0|
|2020-01-20|2020|    1| 20|            Afghanistan|        Asia|              0.0|               0.0|
|2020-01-10|2020|    1| 10|            Afghanistan|        Asia|              0.0|               0.0|
|2020-10-24|2020|   10| 24|                Albania|      Europe|            306.0|               4.0|
|2020-11-30|2020|   11| 30|                Algeria|      Africa|           1009.0|              17.0|
|2020-05-02|2020|    5|  2|                Andorra|      Europe|              1.0|               1.0|
|2020-09-08|2020|    9|  8|                 Angola|      Africa|             30.0|

In [16]:
daily_agg.count()

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 20, Finished, Available, Finished, False)

61900

In [17]:
# Find the minimum and maximum value of the 'daterep' column
df_silver.select(
    F.min("daterep").alias("min_date"),
    F.max("daterep").alias("max_date")
).show()

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 21, Finished, Available, Finished, False)

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2019-12-31|2020-12-14|
+----------+----------+



In [18]:
from pyspark.sql import Window
import pyspark.sql.functions as F

# Define a window specification to calculate a 7-day rolling average
window_spec = Window.partitionBy("countriesAndTerritories").orderBy("daterep").rowsBetween(-6, 0)

# Calculate 7-days moving average for cases and deaths
df_silver = daily_agg.withColumn('7_day_avg_cases', F.avg('total_daily_cases').over(window_spec))
df_silver = df_silver.withColumn('7_day_avg_deaths', F.avg('total_daily_deaths').over(window_spec))

# show the result
df_silver.show()

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 26, Finished, Available, Finished, False)

+----------+----+-----+---+-----------------------+------------+-----------------+------------------+---------------+----------------+
|   daterep|year|month|day|countriesAndTerritories|continentExp|total_daily_cases|total_daily_deaths|7_day_avg_cases|7_day_avg_deaths|
+----------+----+-----+---+-----------------------+------------+-----------------+------------------+---------------+----------------+
|2019-12-31|2019|   12| 31|            Afghanistan|        Asia|              0.0|               0.0|            0.0|             0.0|
|2020-01-01|2020|    1|  1|            Afghanistan|        Asia|              0.0|               0.0|            0.0|             0.0|
|2020-01-02|2020|    1|  2|            Afghanistan|        Asia|              0.0|               0.0|            0.0|             0.0|
|2020-01-03|2020|    1|  3|            Afghanistan|        Asia|              0.0|               0.0|            0.0|             0.0|
|2020-01-04|2020|    1|  4|            Afghanistan|    

In [19]:
from pyspark.sql.window import Window

windowSpec = Window.partitionBy("countriesAndTerritories").orderBy("daterep")

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 28, Finished, Available, Finished, False)

In [20]:
from pyspark.sql.functions import sum

# Calculate cumulative cases
df_silver = df_silver.withColumn("cumulative_cases", sum("total_daily_cases").over(windowSpec))

# Calculate cumulative deaths
df_silver = df_silver.withColumn("cumulative_deaths", sum("total_daily_deaths").over(windowSpec))

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 29, Finished, Available, Finished, False)

In [21]:
display(df_silver)

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 52a5439c-d4c6-4292-9613-75b6bfbb2ca3)

In [22]:
# Store cleaned and transformed data as a Delta table in silver layer
df_silver.write.format("delta").mode("overwrite").save("Tables/dbo/covid_silver_transformed_data")

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 31, Finished, Available, Finished, False)

In [23]:
df_silver.count()

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 32, Finished, Available, Finished, False)

61900

### Load Fact table

In [24]:
from pyspark.sql import functions as F

# Assuming df_transformed in your final transformed dataframe with all columns 
df_transformed = spark.sql("SELECT * FROM Covid_LH.dbo.covid_silver_transformed_data")

# Joining with dim_date (assuming it's based on 'daterep')
dim_date = spark.sql("SELECT * FROM Covid_LH.dbo.dim_date")
fact_table = (
    df_transformed
        .join(dim_date, df_transformed['daterep'] == dim_date['date_key'], 'inner')
        .drop('daterep')
)

# Joining with dim_location (assuming it's based on 'countriesAndTerritories')
dim_location = spark.sql("SELECT * FROM Covid_LH.dbo.dim_location")
fact_table = (
    fact_table
        .join(dim_location, df_transformed['countriesAndTerritories'] == dim_location['countriesAndTerritories'], 'inner')
        .drop('countriesAndTerritories')
)

# Selecting required columns for the fact table
fact_table = fact_table.select(
    dim_date['date_key'],
    dim_location['location_key'],
    df_transformed['total_daily_cases'].alias('daily_cases'),
    df_transformed['total_daily_deaths'].alias('daily_deaths'),
    df_transformed['7_day_avg_cases'].alias('avg_7_day_cases'),
    df_transformed['7_day_avg_deaths'].alias('avg_7_day_deaths'),
    df_transformed['cumulative_cases'].alias('cumulative_cases'),
    df_transformed['cumulative_deaths'].alias('cumulative_deaths'),
    F.current_date().alias('load_date')  # Load date for tracking
)

# write fact table partitioned by 'location_key' and derived 'year' from 'date_key' 
fact_table.write.partitionBy("location_key").format("delta").mode("overwrite").save("Tables/dbo/covid_gold_aggregated_kpis")

StatementMeta(, 3e6cdabf-246c-4d9c-81f7-fc0d184ea7ab, 33, Finished, Available, Finished, False)